# 🎵 Audio HMTL 情绪分类训练 (Wav2Vec 2.0)
## Google Colab GPU 加速版

**使用方法:**
1. 上传 `audio_hmtl_labels.csv` 到 Colab
2. 上传音频数据集到 Google Drive
3. 运行所有单元格
4. 训练完成后下载模型

In [ ]:
# 检查 GPU
!nvidia-smi

In [ ]:
# 安装依赖
!pip install -q transformers librosa soundfile tqdm scikit-learn

In [ ]:
# 挂载 Google Drive (如果数据集在 Drive 中)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Config, Wav2Vec2Processor
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==================== 配置 ====================
# 修改这里：指向你的标签文件和音频目录
LABEL_CSV_PATH = '/content/audio_hmtl_labels.csv'  # 上传到 Colab
# 或者使用 Google Drive:
# LABEL_CSV_PATH = '/content/drive/MyDrive/audio_hmtl_labels.csv'

NUM_EPOCHS = 10
BATCH_SIZE = 16  # GPU 可以用更大的 batch size
LEARNING_RATE = 5e-5
MODEL_SAVE_PATH = '/content/audio_best_hmtl.pt'

In [ ]:
# ==================== AudioPreprocessor ====================
class AudioPreprocessor:
    def __init__(self):
        self.processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base')
        self.sampling_rate = self.processor.feature_extractor.sampling_rate

    def preprocess_audio(self, audio_file_path):
        if not os.path.exists(audio_file_path):
            return None
            
        try:
            speech, rate = librosa.load(audio_file_path, sr=self.sampling_rate)
            
            processed = self.processor(
                speech, 
                sampling_rate=self.sampling_rate, 
                return_tensors="pt", 
                padding=True
            )
            
            input_values = processed.input_values.squeeze(0)
            attention_mask = torch.ones(input_values.shape, dtype=torch.long)
            
            return {
                'input_values': input_values,
                'attention_mask': attention_mask
            }

        except Exception as e:
            return None

In [ ]:
# ==================== AudioHMTLClassifier ====================
class AudioHMTLClassifier(nn.Module):
    def __init__(self):
        super(AudioHMTLClassifier, self).__init__()
        
        self.wav2vec2 = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        self.wav2vec2.freeze_feature_extractor()

        config = Wav2Vec2Config.from_pretrained('facebook/wav2vec2-base')
        hidden_size = config.hidden_size
        
        self.dim_reducer = nn.Linear(hidden_size, 256)
        self.relu = nn.ReLU()
        reduced_size = 256

        # HMTL 分类头
        self.classifier_4 = nn.Linear(reduced_size, 4)  # 4核心分类
        self.classifier_3 = nn.Linear(reduced_size, 3)  # 3极性分类
        self.regressor_A = nn.Linear(reduced_size, 1)   # Arousal 回归
        self.regressor_V = nn.Linear(reduced_size, 1)   # Valence 回归

    def forward(self, input_values, attention_mask):
        output = self.wav2vec2(input_values, attention_mask=attention_mask)
        mean_pooling = torch.mean(output.last_hidden_state, dim=1)
        x = self.relu(self.dim_reducer(mean_pooling))

        logits_4 = self.classifier_4(x)
        logits_3 = self.classifier_3(x)
        pred_A = self.regressor_A(x)
        pred_V = self.regressor_V(x)

        return logits_4, logits_3, pred_A, pred_V

In [ ]:
# ==================== Dataset ====================
class AudioDataset(Dataset):
    def __init__(self, label_df):
        self.label_df = label_df
        self.preprocessor = AudioPreprocessor() 

    def __len__(self):
        return len(self.label_df)

    def __getitem__(self, idx):
        row = self.label_df.iloc[idx]
        audio_full_path = row['audio_full_path']
        
        processed_features = self.preprocessor.preprocess_audio(audio_full_path)
        
        if processed_features is None:
             return None
             
        input_values = processed_features['input_values']
        attention_mask = processed_features['attention_mask']

        labels_4 = torch.tensor(row['label_4_emotion'], dtype=torch.long)
        labels_3 = torch.tensor(row['label_3_polarity'], dtype=torch.long)
        true_A = torch.tensor(row['true_arousal'], dtype=torch.float)
        true_V = torch.tensor(row['true_valence'], dtype=torch.float)

        return {
            'input_values': input_values,
            'attention_mask': attention_mask,
            'labels_4': labels_4,
            'labels_3': labels_3,
            'true_A': true_A,
            'true_V': true_V
        }

def custom_collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return None

    max_len = max(item['input_values'].shape[0] for item in batch)
    
    padded_batch = {}
    for key in batch[0].keys():
        if key in ['input_values', 'attention_mask']:
            padded_tensors = [
                torch.nn.functional.pad(item[key], (0, max_len - item[key].shape[0]), 'constant', 0.0) 
                for item in batch
            ]
            padded_batch[key] = torch.stack(padded_tensors)
        else:
            padded_batch[key] = torch.stack([item[key] for item in batch])
            
    return padded_batch

In [ ]:
# ==================== Loss & Evaluation ====================
def HMTL_Audio_Loss(logits_4, logits_3, pred_A, pred_V, 
                    labels_4, labels_3, true_A, true_V, 
                    lambda_1=0.2, lambda_A=1.5, lambda_V=0.5):
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss() 
    
    L_4_emotion = ce_loss(logits_4, labels_4)
    L_3_polarity = ce_loss(logits_3, labels_3)
    L_arousal = mse_loss(pred_A.squeeze(-1), true_A)
    L_valence = mse_loss(pred_V.squeeze(-1), true_V)
    
    L_total = L_4_emotion + lambda_1 * L_3_polarity + lambda_A * L_arousal + lambda_V * L_valence
    return L_total, L_4_emotion, L_3_polarity, L_arousal, L_valence

def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds_4, all_labels_4 = [], []
    all_preds_A, all_labels_A = [], []
    
    with torch.no_grad():
        for batch in data_loader:
            if batch is None: continue 
            
            input_values = batch['input_values'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_4 = batch['labels_4'].to(device)
            true_A = batch['true_A'].to(device)

            logits_4, _, pred_A, _ = model(input_values, attention_mask)
            
            preds_4 = torch.argmax(logits_4, dim=1).cpu().numpy()
            all_preds_4.extend(preds_4)
            all_labels_4.extend(labels_4.cpu().numpy())
            all_preds_A.extend(pred_A.squeeze(-1).cpu().numpy())
            all_labels_A.extend(true_A.cpu().numpy())

    if len(all_labels_4) == 0:
        model.train()
        return 0.0, 0.0
    
    acc_4 = accuracy_score(all_labels_4, all_preds_4)
    mse_A = mean_squared_error(all_labels_A, all_preds_A)
    model.train()
    return acc_4, mse_A

In [ ]:
# ==================== 加载数据 ====================
label_df = pd.read_csv(LABEL_CSV_PATH)
print(f"✅ 成功加载 {len(label_df)} 条数据")
print(f"\n前3个样本:")
print(label_df.head(3))

train_df, test_df = train_test_split(label_df, test_size=0.2, random_state=42)
train_dataset = AudioDataset(train_df.reset_index(drop=True))
test_dataset = AudioDataset(test_df.reset_index(drop=True))

print(f"\n训练集: {len(train_dataset)} 样本")
print(f"测试集: {len(test_dataset)} 样本")

In [ ]:
# ==================== 初始化模型和训练器 ====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n使用设备: {device}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=2, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                         num_workers=2, collate_fn=custom_collate_fn)

model = AudioHMTLClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"\n模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

In [ ]:
# ==================== 训练循环 ====================
best_acc_4 = 0.0
model.train()

print(f"\n🚀 开始训练 {NUM_EPOCHS} 个 epochs...")

for epoch in range(NUM_EPOCHS):
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in pbar:
        if batch is None: 
            continue
            
        optimizer.zero_grad()
        
        input_values = batch['input_values'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_4 = batch['labels_4'].to(device)
        labels_3 = batch['labels_3'].to(device)
        true_A = batch['true_A'].to(device)
        true_V = batch['true_V'].to(device)

        logits_4, logits_3, pred_A, pred_V = model(input_values, attention_mask)
        L_total, L_4, L_3, L_A, L_V = HMTL_Audio_Loss(
            logits_4, logits_3, pred_A, pred_V, 
            labels_4, labels_3, true_A, true_V
        )
        
        L_total.backward()
        optimizer.step()
        total_loss += L_total.item()
        
        pbar.set_postfix({
            'Loss': f'{L_total.item():.2e}', 
            'L4': f'{L_4.item():.2e}',
            'LA': f'{L_A.item():.2e}',
        })
        
    avg_loss = total_loss / len(train_loader)
    acc_4, mse_A = evaluate_model(model, test_loader, device)
    
    print(f"\nEpoch {epoch+1} 完成:")
    print(f"  → 训练平均损失: {avg_loss:.4e}")
    print(f"  → 测试集 4核心准确率: {acc_4:.4f}")
    print(f"  → 测试集 Arousal MSE: {mse_A:.4f}")

    if acc_4 > best_acc_4:
        best_acc_4 = acc_4
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  ✨ 模型已保存 (准确率: {best_acc_4:.4f})")

print("\n✅ 训练完成！")

In [ ]:
# ==================== 下载模型 ====================
from google.colab import files
files.download(MODEL_SAVE_PATH)
print("\n✅ 模型已下载到本地")